In [1]:
import pandas as pd

df = pd.read_json("../data/function.json", dtype={"commit_id": str})
train = pd.read_json("../data/train.jsonl", lines=True, dtype={"commit_id": str})
test = pd.read_json("../data/test.jsonl", lines=True, dtype={"commit_id": str})

print("functions:", len(df), "| unique commits:", df["commit_id"].nunique())
print(df.groupby("commit_id").size().describe())

train_commits = set(train["commit_id"])
print("test functions whose commit also appears in train:",
      f'{test["commit_id"].isin(train_commits).mean():.1%}')

functions: 27318 | unique commits: 12316
count    12316.000000
mean         2.218090
std          7.406717
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max        588.000000
dtype: float64
test functions whose commit also appears in train: 64.7%


In [2]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, tmp_idx = next(gss.split(df, df["target"], groups=df["commit_id"]))
g_train, g_tmp = df.iloc[tr_idx], df.iloc[tmp_idx]

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
v_idx, t_idx = next(gss2.split(g_tmp, g_tmp["target"], groups=g_tmp["commit_id"]))
g_valid, g_test = g_tmp.iloc[v_idx], g_tmp.iloc[t_idx]

print(len(g_train), len(g_valid), len(g_test))
print(g_train["target"].mean(), g_valid["target"].mean(), g_test["target"].mean())
print("commit overlap train/test:", len(set(g_train.commit_id) & set(g_test.commit_id)))

21895 2972 2451
0.4546700159853848 0.4572678331090175 0.4675642594859241
commit overlap train/test: 0


In [3]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def clean_code(code):
    code = re.sub(r"/\*.*?\*/", " ", code, flags=re.DOTALL)
    code = re.sub(r"//[^\n]*", " ", code)
    return re.sub(r"\s+", " ", code).strip()

for d in (g_train, g_test):
    d["clean"] = d["func"].apply(clean_code)

vec = TfidfVectorizer(
    token_pattern=r"[A-Za-z_][A-Za-z0-9_]*|\d+|==|!=|<=|>=|->|&&|\|\||[^\s\w]",
    lowercase=False, ngram_range=(1, 2), min_df=2, max_features=50000)
Xtr = vec.fit_transform(g_train["clean"])
Xte = vec.transform(g_test["clean"])

models = {
    "Naive Bayes": MultinomialNB(alpha=0.01),
    "SVM": LinearSVC(C=0.5, class_weight="balanced", max_iter=5000),
}
for name, m in models.items():
    p = m.fit(Xtr, g_train["target"]).predict(Xte)
    y = g_test["target"]
    print(f"{name}: acc={accuracy_score(y, p):.3f}, precision={precision_score(y, p):.3f}, "
          f"recall={recall_score(y, p):.3f}, F1={f1_score(y, p):.3f}")

Naive Bayes: acc=0.537, precision=0.504, recall=0.533, F1=0.518
SVM: acc=0.527, precision=0.494, recall=0.542, F1=0.517


In [4]:
g_train.to_json("../data/grouped_train.jsonl", orient="records", lines=True)
g_valid.to_json("../data/grouped_valid.jsonl", orient="records", lines=True)
g_test.to_json("../data/grouped_test.jsonl", orient="records", lines=True)